<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/devoirs/Homework_Chest_XRay_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 2: Estimation d'Âge sur Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

**Note:** /20 points

---

## Objectif

Utiliser un modèle d'IA pré-entraîné pour estimer l'âge de patients à partir de radiographies thoraciques.

## Contexte Clinique

Les radiographies thoraciques contiennent des indices sur l'âge:
- Calcifications vasculaires et valvulaires
- Ostéoporose et densité osseuse  
- Modifications cardiothoraciques
- Changements pulmonaires liés à l'âge

**Applications**: Vérification d'identité, détection d'erreurs, médecine légale

## Barème (/20)

1. Setup (4 pts)
2. Prédictions (8 pts)
3. Évaluation (5 pts)  
4. Réflexion (3 pts)

**Ressources**:
- Doc: https://mlmed.org/torchxrayvision/
- Dataset: https://huggingface.co/datasets/BahaaEldin0/NIH-Chest-Xray-14

## Partie 1: Setup (4 points)

In [ ]:
!pip install -q torchxrayvision torch torchvision datasets matplotlib pandas scikit-learn seaborn

In [ ]:
import torchxrayvision as xrv
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error, r2_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

### Question 1.1 (2 pts): Charger le modèle

Chargez le **Riken AgeModel** (MAE=3.67 ans sur NIH dataset)

In [ ]:
# TODO: Complétez la ligne ci-dessous
model = xrv.baseline_models.riken.AgeModel()
model = model.to(device)
model.eval()

print(f"Modèle chargé!")
print(f"Cible: {model.targets}")

### Question 1.2 (2 pts): Charger le dataset

Chargez le dataset NIH Chest X-ray 14 et sous-échantillonnez 1000 images.

In [ ]:
print("Chargement du dataset (quelques minutes)...")
dataset = load_dataset("BahaaEldin0/NIH-Chest-Xray-14", split="test")
print(f"Dataset complet: {len(dataset)} images")

# TODO: Sous-échantillonnez 1000 images avec seed=42
# Indice: dataset.shuffle(seed=42).select(range(1000))
dataset_sample = dataset.shuffle(seed=42).select(range(1000))

print(f"\nÉchantillon: {len(dataset_sample)} images")
print(f"\nColonnes: {dataset_sample.column_names}")
print(f"\nExemple:")
for key in dataset_sample[0].keys():
    if key != 'image':
        print(f"  {key}: {dataset_sample[0][key]}")

## Partie 2: Prédictions (8 points)

### Question 2.1 (3 pts): Fonction de prédiction

In [ ]:
def predict_age(image, model):
    """Prédit l'âge depuis une image PIL."""
    with torch.no_grad():
        # Convertir PIL en numpy
        img_np = np.array(image)
        
        # TODO: Normalisez l'image avec xrv.datasets.normalize(img_np, 255)
        img = xrv.datasets.normalize(img_np, 255)
        
        # Convertir en grayscale si besoin
        if len(img.shape) > 2:
            img = img[:, :, 0]
        
        # Ajouter dimension canal
        img = img[None, :, :]
        
        # TODO: Convertir en tensor PyTorch
        # Indice: torch.from_numpy(img)
        img_tensor = torch.from_numpy(img).to(device)
        
        # TODO: Ajouter batch dimension
        # Indice: img_tensor.unsqueeze(0)
        img_tensor = img_tensor.unsqueeze(0)
        
        # Prédire
        pred = model(img_tensor)
        age_pred = float(pred[0][0].cpu())
    
    return age_pred

### Question 2.2 (2 pts): Tester sur un exemple

In [ ]:
# Récupérer première image
example = dataset_sample[0]
example_image = example['image']

# TODO: Trouvez le nom de la colonne pour l'âge en examinant 'example' ci-dessus
# Essayez: 'Patient Age', 'age', 'patient_age', etc.
true_age = example['Patient Age']  # Modifiez si le nom est différent

# Prédire
predicted_age = predict_age(example_image, model)
error = predicted_age - true_age

# Afficher
plt.figure(figsize=(6, 6))
plt.imshow(example_image, cmap='gray')
plt.title(f"Âge réel: {true_age} ans\nÂge prédit: {predicted_age:.1f} ans\nErreur: {error:+.1f} ans")
plt.axis('off')
plt.show()

### Question 2.3 (3 pts): Analyser 50 images

**Note**: 50 images = ~2-3 minutes de calcul

In [ ]:
results = []
n_samples = 50

print(f"Analyse de {n_samples} images...")
for i in range(n_samples):
    print(f"\rImage {i+1}/{n_samples}", end="")
    
    # TODO: Récupérez l'image et l'âge réel
    item = dataset_sample[i]
    image = item['image']
    true_age = item['Patient Age']  # Adaptez le nom si différent
    
    # TODO: Prédisez l'âge
    predicted_age = predict_age(image, model)
    
    # Stocker
    results.append({
        'age_reel': true_age,
        'age_predit': predicted_age,
        'erreur': predicted_age - true_age,
        'erreur_abs': abs(predicted_age - true_age)
    })

print("\nTerminé!")
results_df = pd.DataFrame(results)
print("\nPremiers résultats:")
print(results_df.head(10))

## Partie 3: Évaluation (5 points)

### Question 3.1 (2 pts): Métriques

In [ ]:
# TODO: Calculez les métriques
mae = mean_absolute_error(results_df['age_reel'], results_df['age_predit'])
median_error = results_df['erreur_abs'].median()
r2 = r2_score(results_df['age_reel'], results_df['age_predit'])
percent_within_5 = (results_df['erreur_abs'] <= 5).mean() * 100

print("\nMÉTRIQUES")
print("="*50)
print(f"MAE: {mae:.2f} ans")
print(f"Erreur médiane: {median_error:.2f} ans")
print(f"R²: {r2:.3f}")
print(f"Prédictions à ±5 ans: {percent_within_5:.1f}%")
print("="*50)
print(f"\nRéférence paper: MAE = 3.67 ans")
print(f"Votre performance: {'Excellente' if mae < 5 else 'Bonne' if mae < 8 else 'À améliorer'}")

### Question 3.2 (2 pts): Visualisations

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Scatter plot Âge réel vs prédit
ax1.scatter(results_df['age_reel'], results_df['age_predit'], alpha=0.6, s=80, edgecolors='black')
min_age = min(results_df['age_reel'].min(), results_df['age_predit'].min())
max_age = max(results_df['age_reel'].max(), results_df['age_predit'].max())
ax1.plot([min_age, max_age], [min_age, max_age], 'r--', lw=2, label='Parfait')
ax1.set_xlabel('Âge Réel (ans)', fontsize=12)
ax1.set_ylabel('Âge Prédit (ans)', fontsize=12)
ax1.set_title(f'Âge Réel vs Prédit (MAE={mae:.2f} ans)', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# TODO: Histogramme des erreurs
ax2.hist(results_df['erreur'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax2.axvline(0, color='red', linestyle='--', lw=2, label='Erreur=0')
ax2.set_xlabel('Erreur (ans)', fontsize=12)
ax2.set_ylabel('Fréquence', fontsize=12)
ax2.set_title('Distribution des Erreurs', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Question 3.3 (1 pt): Analyse par groupe d'âge

In [ ]:
# TODO: Créez une fonction qui catégorise les âges
def categorize_age(age):
    if age < 30:
        return '<30'
    elif age < 50:
        return '30-50'
    elif age < 70:
        return '50-70'
    else:
        return '>70'

results_df['groupe'] = results_df['age_reel'].apply(categorize_age)

# Calculer MAE par groupe
print("\nMAE par groupe d'âge:")
for groupe in ['<30', '30-50', '50-70', '>70']:
    subset = results_df[results_df['groupe'] == groupe]
    if len(subset) > 0:
        mae_groupe = subset['erreur_abs'].mean()
        print(f"{groupe:8s}: MAE={mae_groupe:.2f} ans (n={len(subset)})")

## Partie 4: Réflexion (3 points)

### Question 4.1 (1 pt): Applications

Rédigez 5-7 lignes:
1. Cas cliniques où l'estimation d'âge serait utile?
2. MAE de 3-4 ans est-elle suffisante?

**Réponse**: [Écrivez ici]

### Question 4.2 (1 pt): Limites

Listez 3 limites:

1. [Limite 1]
2. [Limite 2]  
3. [Limite 3]

### Question 4.3 (1 pt): Biais

Modèle entraîné sur patients américains. Quels biais potentiels? (3-5 lignes)

**Réponse**: [Écrivez ici]

## BONUS: Testez votre propre image!

In [ ]:
from google.colab import files
import io

print("Uploadez une radiographie thoracique:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    image = Image.open(io.BytesIO(uploaded[filename]))
    predicted_age = predict_age(image, model)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image, cmap='gray')
    plt.title(f"Âge prédit: {predicted_age:.1f} ans", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print(f"\nÂge estimé: {predicted_age:.1f} ans")
    print(f"IC approximatif: [{predicted_age-3.7:.1f}, {predicted_age+3.7:.1f}] ans")

## Remise

1. Exécution → Tout exécuter
2. Vérifiez les erreurs
3. Complétez les réponses textuelles
4. Sauvegardez dans Drive
5. Partagez le lien

**Bon travail!**